# Stellar Luminosity: Linear and Polynomial Regression from First Principles

This notebook studies how a regression model learns from data. The goal is to implement prediction, loss, gradients, and gradient descent directly with NumPy, then compare a linear model with a polynomial model for stellar mass and luminosity.

## 1. Environment Setup

The project uses only Python, NumPy, and Matplotlib.

In [ ]:
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

print(f"Python version: {sys.version.split()[0]}")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")

## 2. Dataset Loading

The observations describe stellar mass in solar masses and luminosity in solar luminosities. This is an instructional dataset, so it is useful for learning regression but not enough for scientific conclusions.

In [ ]:
mass = np.array([0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4], dtype=float)
luminosity = np.array([0.15, 0.35, 1.00, 2.30, 4.10, 7.00, 11.2, 17.5, 25.0, 35.0], dtype=float)

print(f"mass shape: {mass.shape}")
print(f"luminosity shape: {luminosity.shape}")
print(f"mass range: {mass.min()} to {mass.max()}")
print(f"luminosity range: {luminosity.min()} to {luminosity.max()}")

## 3. Explore Before Modeling

Before training a model, it is important to inspect the shape of the data.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(mass, luminosity, color="tab:blue", label="Observed data")
plt.xlabel("Stellar mass (solar masses)")
plt.ylabel("Luminosity (solar luminosities)")
plt.title("Stellar Mass vs Luminosity")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

The relationship is increasing and clearly nonlinear. Luminosity grows slowly at lower masses and much faster at higher masses. A straight line may capture the general upward direction, but it is likely to make systematic errors: it may overestimate some lower values and underestimate the rapid growth at the upper end of the observed range.

## 4. Vectorized Regression Functions

The functions below support design matrices with one or more features. The same implementation will be reused for the linear and polynomial models.

In [ ]:
def predict(X, w, b):
    """Return model predictions for a design matrix X."""
    return X @ w + b


def compute_cost(X, y, w, b):
    """Compute mean squared error using J = (1 / 2m) * sum(error^2)."""
    m = X.shape[0]
    error = predict(X, w, b) - y
    return (1 / (2 * m)) * np.sum(error ** 2)


def compute_mse(X, y, w, b):
    """Compute the standard mean squared error for model evaluation."""
    error = predict(X, w, b) - y
    return np.mean(error ** 2)


def compute_gradient(X, y, w, b):
    """Compute vectorized gradients for w and b."""
    m = X.shape[0]
    error = predict(X, w, b) - y
    dj_dw = (1 / m) * (X.T @ error)
    dj_db = (1 / m) * np.sum(error)
    return dj_dw, dj_db


def gradient_descent(X, y, w_init, b_init, alpha, num_iters):
    """Run gradient descent and return learned parameters plus cost history."""
    w = w_init.copy()
    b = float(b_init)
    cost_history = []
    
    for _ in range(num_iters):
        dj_dw, dj_db = compute_gradient(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        cost_history.append(compute_cost(X, y, w, b))
    
    return w, b, np.array(cost_history)

## 5. Train a Linear Model

The linear representation uses one feature: stellar mass.

In [ ]:
X_linear = mass.reshape(-1, 1)

w_linear_init = np.zeros(X_linear.shape[1])
b_linear_init = 0.0
alpha_linear = 0.05
iterations_linear = 5000

w_linear, b_linear, cost_linear_history = gradient_descent(
    X_linear,
    luminosity,
    w_linear_init,
    b_linear_init,
    alpha_linear,
    iterations_linear,
)

final_cost_linear = compute_cost(X_linear, luminosity, w_linear, b_linear)
final_mse_linear = compute_mse(X_linear, luminosity, w_linear, b_linear)

print(f"Linear weights: {w_linear}")
print(f"Linear bias: {b_linear:.6f}")
print(f"Linear final cost: {final_cost_linear:.6f}")
print(f"Linear final MSE: {final_mse_linear:.6f}")

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(cost_linear_history, color="tab:orange")
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.title("Linear Model Convergence")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
mass_grid = np.linspace(mass.min(), mass.max(), 200)
X_linear_grid = mass_grid.reshape(-1, 1)
linear_grid_predictions = predict(X_linear_grid, w_linear, b_linear)

plt.figure(figsize=(7, 5))
plt.scatter(mass, luminosity, color="tab:blue", label="Observed data")
plt.plot(mass_grid, linear_grid_predictions, color="tab:orange", label="Linear fit")
plt.xlabel("Stellar mass (solar masses)")
plt.ylabel("Luminosity (solar luminosities)")
plt.title("Linear Regression Fit")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

The linear model converges when the cost curve decreases and then becomes nearly flat. However, the fitted line cannot follow the accelerating growth in the observations. This creates systematic residuals because the model representation is too simple for the observed pattern.

## 6. Train a Polynomial Model

The polynomial representation uses two features: mass and mass squared. The learning algorithm remains unchanged.

In [ ]:
X_polynomial = np.column_stack((mass, mass ** 2))

w_poly_init = np.zeros(X_polynomial.shape[1])
b_poly_init = 0.0
alpha_poly = 0.01
iterations_poly = 10000

w_poly, b_poly, cost_poly_history = gradient_descent(
    X_polynomial,
    luminosity,
    w_poly_init,
    b_poly_init,
    alpha_poly,
    iterations_poly,
)

final_cost_poly = compute_cost(X_polynomial, luminosity, w_poly, b_poly)
final_mse_poly = compute_mse(X_polynomial, luminosity, w_poly, b_poly)

print(f"Polynomial weights: {w_poly}")
print(f"Polynomial bias: {b_poly:.6f}")
print(f"Polynomial final cost: {final_cost_poly:.6f}")
print(f"Polynomial final MSE: {final_mse_poly:.6f}")

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(cost_poly_history, color="tab:green")
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.title("Polynomial Model Convergence")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
X_poly_grid = np.column_stack((mass_grid, mass_grid ** 2))
poly_grid_predictions = predict(X_poly_grid, w_poly, b_poly)

plt.figure(figsize=(7, 5))
plt.scatter(mass, luminosity, color="tab:blue", label="Observed data")
plt.plot(mass_grid, poly_grid_predictions, color="tab:green", label="Polynomial fit")
plt.xlabel("Stellar mass (solar masses)")
plt.ylabel("Luminosity (solar luminosities)")
plt.title("Polynomial Regression Fit")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

The model changed because the input representation now includes an additional feature, mass squared. The prediction, cost, gradient, and gradient descent functions remained the same. This shows that feature engineering can change what patterns a model can represent without changing the optimization algorithm.

## 7. Compare the Models

The models can be compared with final cost, fitted curves, and residuals.

In [ ]:
linear_predictions = predict(X_linear, w_linear, b_linear)
poly_predictions = predict(X_polynomial, w_poly, b_poly)

linear_residuals = luminosity - linear_predictions
poly_residuals = luminosity - poly_predictions

print(f"Linear final cost: {final_cost_linear:.6f}")
print(f"Linear final MSE: {final_mse_linear:.6f}")
print(f"Polynomial final cost: {final_cost_poly:.6f}")
print(f"Polynomial final MSE: {final_mse_poly:.6f}")

comparison_table = np.column_stack((mass, luminosity, linear_predictions, poly_predictions, linear_residuals, poly_residuals))
print("\nColumns: mass, actual, linear_pred, poly_pred, linear_residual, poly_residual")
print(np.round(comparison_table, 4))

In [ ]:
plt.figure(figsize=(8, 5))
plt.axhline(0, color="black", linewidth=1)
plt.scatter(mass, linear_residuals, color="tab:orange", label="Linear residuals")
plt.scatter(mass, poly_residuals, color="tab:green", label="Polynomial residuals")
plt.xlabel("Stellar mass (solar masses)")
plt.ylabel("Residual: actual - predicted")
plt.title("Residual Comparison")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(mass, luminosity, color="tab:blue", label="Observed data")
plt.plot(mass_grid, linear_grid_predictions, color="tab:orange", label="Linear fit")
plt.plot(mass_grid, poly_grid_predictions, color="tab:green", label="Polynomial fit")
plt.xlabel("Stellar mass (solar masses)")
plt.ylabel("Luminosity (solar luminosities)")
plt.title("Linear vs Polynomial Regression")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

The polynomial model represents the observed range better because the dataset has an accelerating upward curve, and the mass squared feature allows curved predictions. The linear model is limited to a straight line, so it cannot capture that acceleration. Still, lower training error alone does not prove that the polynomial model is scientifically valid; it only shows that this representation fits this small dataset better.

## 8. Test the Boundary of the Evidence

Now both models are tested at one point inside the observed range and one point far outside it.

In [ ]:
test_mass = np.array([1.3, 5.0])
X_test_linear = test_mass.reshape(-1, 1)
X_test_poly = np.column_stack((test_mass, test_mass ** 2))

test_linear_predictions = predict(X_test_linear, w_linear, b_linear)
test_poly_predictions = predict(X_test_poly, w_poly, b_poly)

for m, linear_pred, poly_pred in zip(test_mass, test_linear_predictions, test_poly_predictions):
    location = "inside" if mass.min() <= m <= mass.max() else "outside"
    print(f"mass = {m:.1f} ({location} observed range)")
    print(f"  linear prediction: {linear_pred:.4f}")
    print(f"  polynomial prediction: {poly_pred:.4f}")

The prediction for mass = 1.3 is interpolation because it is inside the observed mass range from 0.6 to 2.4. It is more justified than the prediction for mass = 5.0, which is extrapolation far outside the available evidence. The dataset does not justify strong confidence in either model at mass = 5.0, especially because the true physical relationship may require different theory, more features, and more observations.

## 9. Reflection on Artificial Intelligence

**What does this regression experiment reveal, and what does it fail to reveal, about how more advanced AI systems learn?**

This experiment reveals the basic structure of learning from data: a model makes predictions, measures error, computes gradients, and updates parameters to reduce that error. It also shows that the choice of representation matters because the same learning algorithm can behave differently when the input features change. However, this experiment does not reveal the full complexity of advanced AI systems. Modern systems often learn from massive datasets, high-dimensional representations, nonlinear architectures, and objectives that are much richer than fitting a small curve.

**Do you think Artificial General Intelligence is achievable? Explain what you understand by AGI and why you believe it is or is not reachable.**

I understand AGI as an AI system that can learn, reason, adapt, and act across many domains with a level of flexibility similar to human general intelligence. I think AGI may be achievable in principle, but it is not simply the same as making a larger version of a current model. A generally intelligent system would need robust reasoning, memory, planning, grounding, and the ability to learn from changing situations.

**Could AGI emerge primarily by scaling neural networks with more GPUs, data, and parameters? Or will it require fundamentally different architectures, learning mechanisms, theories, or ways of interacting with the world?**

Scaling has clearly improved AI capabilities, but I do not think scaling alone is guaranteed to produce AGI. More compute, data, and parameters can improve pattern recognition and language behavior, but general intelligence may also require better learning mechanisms, stronger world models, interaction with real environments, long-term memory, and architectures that can reason and verify their own outputs more reliably.

**One question about AI-driven systems that I want to explore during the semester is:**

How can AI-driven systems combine learned patterns with explicit reasoning so that their decisions are more reliable, explainable, and useful in real-world enterprise architectures?